# Evaluación de fusión VIS+IR con YOLO (Ultralytics) + Roboflow

**TFM — Fusión IR/VIS por Morfología Matemática.** Compara el efecto de cada método de fusión sobre la detección de objetos: entrena un YOLO por modalidad/fusión y reporta **mAP@0.5, mAP@0.5:0.95, Precision, Recall, F1**, curvas de pérdida y métricas de **producción** (FPS, parámetros, tamaño, ONNX). 

> Ejecutar en **Google Colab** con `Entorno de ejecución → Cambiar tipo → GPU`. 
> Requiere un dataset etiquetado VIS/IR (Roboflow o M3FD). El TNO no trae cajas.


## 1. Instalación


In [ ]:
!pip -q install ultralytics roboflow opencv-python-headless
import ultralytics, torch; ultralytics.checks()
print('CUDA disponible:', torch.cuda.is_available())


## 2. Código de fusión Top-Hat
Clona tu repositorio para usar `src/fusion/prop_top_hat.py` (o sube la carpeta `src/`).


In [ ]:
# Opción A: clonar tu repo (ajusta la URL)
# !git clone https://github.com/<usuario>/tesis_mciencias_datos.git
# %cd tesis_mciencias_datos

# Opción B: si subiste src/ a /content, agrégalo al path
import sys; sys.path.append('/content')
from src.fusion.prop_top_hat import TopHatFusion
from src.fusion.comparatives import average_fusion, laplacian_pyramid_fusion


## 3. Dataset etiquetado desde Roboflow
Crea un proyecto de **Object Detection** en Roboflow con tus pares (clases p. ej. `person`, `car`), exporta en formato **YOLOv8** y pega tu API key. El dataset debe traer las imágenes VIS (y, si las tienes, IR).


In [ ]:
from roboflow import Roboflow
rf = Roboflow(api_key='TU_API_KEY')
project = rf.workspace('TU_WORKSPACE').project('TU_PROYECTO')
dataset = project.version(1).download('yolov8')
print('Dataset en:', dataset.location)  # contiene data.yaml, train/ valid/ test/


## 4. Generar versiones fusionadas conservando las etiquetas
Las cajas son válidas para todas las versiones porque las imágenes están registradas. Para cada imagen VIS con su par IR se generan las fusiones y se **copian las mismas etiquetas**. 
Ajusta `pair_ir_path()` según cómo nombres tus pares IR.


In [ ]:
import os, shutil, cv2, numpy as np, yaml
from pathlib import Path
BASE = Path(dataset.location)

def load_gray(p):
    im = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    return im.astype(np.float32)/255.0

def pair_ir_path(vis_path):
    # EJEMPLO: IR en carpeta paralela con mismo nombre. Ajusta a tu convención.
    return Path(str(vis_path).replace('/images/', '/images_ir/'))

def save_gray(arr, p):
    Path(p).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(p), (np.clip(arr,0,1)*255).astype('uint8'))

FUSERS = {
    'VIS':    lambda v,i: v,
    'IR':     lambda v,i: i,
    'Fused_WTH_diskL5': lambda v,i: TopHatFusion('disk',5).fuse(v,i),
    'Fused_Laplace':    lambda v,i: laplacian_pyramid_fusion(v,i,4),
}

def build_variant(name, fn):
    root = BASE.parent / f'ds_{name}'
    for split in ['train','valid','test']:
        src_img = BASE/split/'images'; src_lbl = BASE/split/'labels'
        if not src_img.exists(): continue
        for img in src_img.glob('*.*'):
            v = load_gray(img)
            ir_p = pair_ir_path(img)
            i = load_gray(ir_p) if ir_p.exists() else v
            if i.shape != v.shape: i = cv2.resize(i, (v.shape[1], v.shape[0]))
            out = fn(v, i)
            save_gray(out, root/split/'images'/img.name)
            lbl = src_lbl/(img.stem+'.txt')
            if lbl.exists():
                (root/split/'labels').mkdir(parents=True, exist_ok=True)
                shutil.copy(lbl, root/split/'labels'/lbl.name)
    # data.yaml
    with open(BASE/'data.yaml') as f: y = yaml.safe_load(f)
    y['path']=str(root); y['train']='train/images'; y['val']='valid/images'; y['test']='test/images'
    with open(root/'data.yaml','w') as f: yaml.safe_dump(y,f)
    return root/'data.yaml'

data_yamls = {name: str(build_variant(name, fn)) for name,fn in FUSERS.items()}
data_yamls


## 5. Entrenamiento (un YOLO por modalidad/fusión)
Mismos hiperparámetros y semilla para una comparación justa.


In [ ]:
from ultralytics import YOLO
import pandas as pd
EPOCHS, IMGSZ, SEED = 50, 640, 0
results = {}
for name, dyaml in data_yamls.items():
    model = YOLO('yolov8n.pt')
    model.train(data=dyaml, epochs=EPOCHS, imgsz=IMGSZ, seed=SEED,
                project='runs_fusion', name=name, exist_ok=True, verbose=False)
    m = model.val(data=dyaml, split='test')
    P, R = m.box.mp, m.box.mr
    f1 = 2*P*R/(P+R+1e-9)
    results[name] = {'mAP50': m.box.map50, 'mAP50_95': m.box.map,
                     'precision': P, 'recall': R, 'F1': f1,
                     'speed_ms': sum(m.speed.values())}
df = pd.DataFrame(results).T.round(4); df['FPS']=(1000/df['speed_ms']).round(1)
df.to_csv('runs_fusion/metricas_entrenamiento.csv')
df


## 6. Curvas de entrenamiento y comparación de mAP


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(14,5))
df[['mAP50','mAP50_95','precision','recall','F1']].plot.bar(ax=ax[0]); ax[0].set_title('Métricas por modalidad/fusión'); ax[0].set_ylim(0,1)
df['FPS'].plot.bar(ax=ax[1], color='#27ae60'); ax[1].set_title('FPS (producción)')
plt.tight_layout(); plt.savefig('runs_fusion/comparacion.png', dpi=150); plt.show()
# Las curvas de pérdida por modelo están en runs_fusion/<name>/results.png


## 7. Producción: parámetros, FLOPs, ONNX y tamaño


In [ ]:
import os
for name in data_yamls:
    w = f'runs_fusion/{name}/weights/best.pt'
    if not os.path.exists(w): continue
    mdl = YOLO(w)
    info = mdl.info(verbose=False)  # (layers, params, gradients, GFLOPs)
    onnx = mdl.export(format='onnx', imgsz=IMGSZ)
    mb = os.path.getsize(onnx)/1e6
    print(f'{name:20s} params={info[1]:,} GFLOPs={info[3]:.1f} onnx={mb:.1f} MB')


## 8. Confiabilidad estadística
Friedman + Wilcoxon (Holm) sobre la métrica por imagen (p. ej. nº de TP o IoU medio por imagen). Reutiliza `experiments/detection/stats_detection.py` exportando una métrica por imagen y modalidad, o la confianza media por imagen del set de test.


In [ ]:
# Ejemplo: detectabilidad por imagen con el mejor modelo de cada modalidad
from scipy import stats
# construir DataFrame [imagen x modalidad] de una métrica por imagen y aplicar:
# stats.friedmanchisquare(*columnas)  y  stats.wilcoxon(a,b) con corrección de Holm
print('Ver experiments/detection/stats_detection.py para el patrón completo.')
